# N5 — Síntesis de la narrativa cultural (Procesamiento)
## AI-MELT: Nivel 5 — Narrativa cultural

**Propósito:** Identificar el régimen dominante del campo metafórico y formular la narrativa cultural que este configura: la matriz simbólica que orienta el comportamiento colectivo de la comunidad discursiva.

**Entrada:**
- `n4_regimenes.csv`, `n4_escenario_regimen.csv`, `n4_ejes_valorativos.csv`, `n4_metaforas_derivadas.csv`
- `n3_escenarios.csv`, `n3_sesgo_evaluativo.csv`, `n3_afecto.csv`, `n3_posicionamiento.csv`
- `n2_metaforas_convencionales.csv`, `n2_primaria_convencional.csv`
- `n1_metaforas_primarias.csv`

**Salida:**
- `n5_narrativa_cultural.csv` — narrativa dominante con score y métricas
- `n5_narrativas_alternativas.csv` — todas las narrativas (dominante, challenger, emergente, periférica)
- `n5_resumen_campo_metaforico.json` — resumen completo del campo para dashboard
- `n5_metadata.json`

**Visualización:** Ejecutar `N5_narrativa_cultural_viz.ipynb` después de este notebook.

---

### Estructura
1. Configuración
2. Carga de datos de todos los niveles
3. Ranking multicriterio de regímenes
4. Estimación de costo
5. Síntesis de la narrativa dominante (LLM)
6. Narrativas alternativas (LLM)
7. Muestra human-in-the-loop
8. Exportación

## 1. Configuración

In [ ]:
# ============================================================
# 1. DEPENDENCIAS E IMPORTS
# ============================================================
# Descomenta si necesitas instalar:
# !pip install anthropic openai pandas tqdm

import datetime
import importlib
import json
import os
import re
import time
import warnings

import anthropic
import openai
import pandas as pd

warnings.filterwarnings('ignore')

print("✓ Imports completados")

In [ ]:
# ============================================================
# 1B. CONFIGURACIÓN
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  MODIFICA ESTA SECCIÓN SEGÚN TUS NECESIDADES            │
# └─────────────────────────────────────────────────────────┘

# API Keys
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-...")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")

# Rutas
N1_DIR = "./outputs/N1/"
N2_DIR = "./outputs/N2/"
N3_DIR = "./outputs/N3/"
N4_DIR = "./outputs/N4/"
OUTPUT_DIR = "./outputs/N5/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LLM
LLM_PROVIDER = "claude"
CLAUDE_MODEL = "claude-sonnet-4-20250514"
OPENAI_MODEL = "gpt-4o"
TEMPERATURE = 0.3          # Ligeramente más alta: tarea interpretativa/sintética
MAX_RETRIES = 3
RATE_LIMIT_PAUSE = 2.0

# Pesos del ranking multicriterio (deben sumar 1.0)
W_FRECUENCIA = 0.4         # Peso de la frecuencia agregada
W_COHERENCIA = 0.3         # Peso de la coherencia interna
W_DISPERSION = 0.3         # Peso de la distribución textual

# Cuántas narrativas alternativas generar (además de la dominante)
N_ALTERNATIVAS = 2          # challenger + emergente

print("✓ Configuración cargada")
print(f"  LLM: {LLM_PROVIDER}")
print(f"  Pesos ranking: freq={W_FRECUENCIA}, coh={W_COHERENCIA}, disp={W_DISPERSION}")
print(f"  Narrativas a generar: 1 dominante + {N_ALTERNATIVAS} alternativas")

## 2. Carga de datos de todos los niveles

In [ ]:
# ============================================================
# 2. CARGA DE DATOS (TODOS LOS NIVELES)
# ============================================================

def safe_load(path, name):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {name}: {len(df):,} filas")
        return df
    print(f"  ⚠ {name}: no encontrado")
    return pd.DataFrame()

print("Cargando datos de N1 a N4...")

# N4
df_reg = safe_load(os.path.join(N4_DIR, "n4_regimenes.csv"), "n4_regimenes")
df_esc_reg = safe_load(os.path.join(N4_DIR, "n4_escenario_regimen.csv"), "n4_escenario_regimen")
df_ejes = safe_load(os.path.join(N4_DIR, "n4_ejes_valorativos.csv"), "n4_ejes_valorativos")
df_deriv = safe_load(os.path.join(N4_DIR, "n4_metaforas_derivadas.csv"), "n4_metaforas_derivadas")

# N3
df_esc = safe_load(os.path.join(N3_DIR, "n3_escenarios.csv"), "n3_escenarios")
df_sesgo = safe_load(os.path.join(N3_DIR, "n3_sesgo_evaluativo.csv"), "n3_sesgo_evaluativo")
df_afecto = safe_load(os.path.join(N3_DIR, "n3_afecto.csv"), "n3_afecto")
df_pos = safe_load(os.path.join(N3_DIR, "n3_posicionamiento.csv"), "n3_posicionamiento")
df_narr = safe_load(os.path.join(N3_DIR, "n3_secuencia_narrativa.csv"), "n3_secuencia_narrativa")

# N2
df_conv = safe_load(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"), "n2_metaforas_convencionales")
df_mn = safe_load(os.path.join(N2_DIR, "n2_primaria_convencional.csv"), "n2_primaria_convencional")

# N1 (solo frecuencias)
df_n1 = safe_load(os.path.join(N1_DIR, "n1_metaforas_primarias.csv"), "n1_metaforas_primarias")

print("\n✓ Todos los datos cargados")

## 3. Ranking multicriterio de regímenes

Fórmula: `score = w1 × freq_norm + w2 × coherencia + w3 × dispersión`

Donde:
- **freq_norm:** frecuencia agregada normalizada (0-1) — suma de instancias de metáforas primarias que alimentan el régimen
- **coherencia:** similitud coseno promedio entre sesgos de sus escenarios (ya calculada en N4)
- **dispersión:** coeficiente de dispersión textual a lo largo del corpus

In [ ]:
# ============================================================
# 3. RANKING MULTICRITERIO
# ============================================================

ranking_rows = []

for _, reg in df_reg.iterrows():
    rid = reg["ID_regimen"]

    # --- Frecuencia agregada ---
    # Escenarios de este régimen
    esc_ids = df_esc_reg[df_esc_reg["ID_regimen"] == rid]["ID_escenario"].tolist()

    # Metáforas convencionales de esos escenarios
    mc_ids = df_esc[df_esc["ID_escenario"].isin(esc_ids)]["ID_metaconvencional_base"].dropna().unique()

    # Metáforas primarias de esas convencionales (vía M:N)
    prim_ids = df_mn[df_mn["ID_metaconvencional"].isin(mc_ids)]["ID_expresion"].unique()
    freq_agregada = len(prim_ids)

    # --- Coherencia interna ---
    coherencia = reg.get("coherencia_interna", 0) or 0

    # --- Dispersión textual ---
    if len(df_n1) > 0 and len(prim_ids) > 0:
        prims_data = df_n1[df_n1["ID_expresion"].isin(prim_ids)]
        n_docs_total = df_n1["ID_documento"].nunique()
        n_docs_regime = prims_data["ID_documento"].nunique()
        dispersion = n_docs_regime / n_docs_total if n_docs_total > 0 else 0
    else:
        dispersion = 0

    ranking_rows.append({
        "ID_regimen": rid,
        "nombre_regimen": reg["nombre_regimen"],
        "n_escenarios": reg["n_escenarios"],
        "frecuencia_agregada": freq_agregada,
        "coherencia_interna": round(float(coherencia), 3),
        "dispersion_textual": round(float(dispersion), 3),
        "n_metaforas_primarias": len(prim_ids),
        "n_documentos": n_docs_regime if len(df_n1) > 0 else 0,
    })

df_ranking = pd.DataFrame(ranking_rows)

# Normalizar frecuencia a [0, 1]
max_freq = df_ranking["frecuencia_agregada"].max()
df_ranking["freq_norm"] = df_ranking["frecuencia_agregada"] / max_freq if max_freq > 0 else 0

# Calcular score compuesto
df_ranking["score"] = (
    W_FRECUENCIA * df_ranking["freq_norm"] +
    W_COHERENCIA * df_ranking["coherencia_interna"] +
    W_DISPERSION * df_ranking["dispersion_textual"]
)

# Clasificar tipo
df_ranking = df_ranking.sort_values("score", ascending=False).reset_index(drop=True)
def assign_type(rank):
    if rank == 0:
        return "dominante"
    elif rank <= N_ALTERNATIVAS:
        return "challenger" if rank == 1 else "emergente"
    return "periferico"

df_ranking["tipo"] = [assign_type(i) for i in range(len(df_ranking))]

print("RANKING DE REGÍMENES")
print("=" * 80)
print(f"  Pesos: freq={W_FRECUENCIA}, coherencia={W_COHERENCIA}, dispersión={W_DISPERSION}")
print()
for _, row in df_ranking.iterrows():
    tipo_icon = {"dominante": "👑", "challenger": "⚔️", "emergente": "🌱", "periferico": "·"}.get(row["tipo"], "")
    print(f"  {tipo_icon} {row['tipo']:12s} | score={row['score']:.3f} | "
          f"freq={row['frecuencia_agregada']:4d} | coh={row['coherencia_interna']:.3f} | "
          f"disp={row['dispersion_textual']:.3f} | {row['nombre_regimen'][:50]}")

dominant_regime = df_ranking.iloc[0]
print(f"\n→ Régimen dominante: {dominant_regime['nombre_regimen']}")

## 4. Estimación de costo antes de procesar

In [ ]:
# ============================================================
# 4. ESTIMACIÓN DE COSTO
# ============================================================

n_calls = 1 + N_ALTERNATIVAS  # dominante + alternativas
est_tok_in = n_calls * 3000    # Prompts largos con toda la estructura
est_tok_out = n_calls * 1500

if LLM_PROVIDER == "claude":
    est_cost = (est_tok_in / 1e6 * 3) + (est_tok_out / 1e6 * 15)
else:
    est_cost = (est_tok_in / 1e6 * 2.5) + (est_tok_out / 1e6 * 10)

est_time = n_calls * (RATE_LIMIT_PAUSE + 5)  # ~5s por llamada (prompts largos)

print("ESTIMACIÓN DE COSTO")
print("=" * 50)
print(f"  Llamadas API: {n_calls} (1 dominante + {N_ALTERNATIVAS} alternativas)")
print(f"  Tokens estimados: ~{est_tok_in:,} input + ~{est_tok_out:,} output")
print(f"  Costo estimado: ${est_cost:.2f} USD ({LLM_PROVIDER})")
print(f"  Tiempo estimado: ~{est_time/60:.1f} minutos")

## 5. Síntesis de la narrativa dominante (LLM)

Se envía al LLM la estructura completa del régimen dominante: escenarios con sus componentes, ejes valorativos, metáforas derivadas y estadísticas.

In [ ]:
# ============================================================
# 5A. PROMPT Y CLIENTE LLM
# ============================================================

if LLM_PROVIDER == "claude":
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    MODEL = CLAUDE_MODEL
else:
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    MODEL = OPENAI_MODEL

SYSTEM_PROMPT_N5 = """Eres un analista del discurso experto en la Teoría de la Metáfora Conceptual y en el modelo MELT (Metaphor Field-Loop Theory). Tu tarea es formular la NARRATIVA CULTURAL que emerge de un régimen de metáforas identificado en el Informe Final de la Comisión de la Verdad de Colombia.

En MELT, la narrativa cultural es una MATRIZ SIMBÓLICA codificada moral y estéticamente, en proceso de configuración permanente, que orienta el comportamiento de los individuos y las comunidades al significar la relación imaginaria entre estos y sus condiciones materiales de existencia en un contexto histórico-espacial determinado (Valdivia, 2024).

Tu formulación debe:
1. Capturar la orientación axiológica del régimen (qué valores legitima, cuáles deslegitima)
2. Identificar las emociones que la narrativa facilita e inhibe
3. Explicar cómo la narrativa significa la relación entre la comunidad y sus condiciones materiales
4. Articular las metáforas, escenarios y ejes valorativos en una síntesis interpretativa coherente

Responde SOLO con JSON válido, sin texto adicional.
"""

PROMPT_NARRATIVA = """Formula la narrativa cultural que configura el siguiente régimen de metáforas del Informe Final de la Comisión de la Verdad de Colombia.

RÉGIMEN: {nombre_regimen}
TIPO: {tipo} (score de dominancia: {score:.3f})
FRECUENCIA AGREGADA: {frecuencia} instancias en {n_docs} documentos
COHERENCIA INTERNA: {coherencia:.3f}
DISPERSIÓN TEXTUAL: {dispersion:.3f}

ESCENARIOS CONSTITUYENTES:
{escenarios_text}

EJES VALORATIVOS:
{ejes_text}

METÁFORAS DERIVADAS POR TRANSITIVIDAD:
{derivadas_text}

POSICIONAMIENTO DE ACTORES SOCIALES:
{posicionamiento_text}

AFECTOS FACILITADOS E INHIBIDOS:
{afectos_text}

Formula la narrativa cultural con esta estructura JSON:
{{
  "nombre_narrativa": "nombre breve y evocador de la narrativa (5-10 palabras)",
  "descripcion_narrativa": "descripción de la matriz simbólica: qué orientación del comportamiento colectivo proyecta, qué valores legitima, qué emociones facilita, cómo significa la relación entre la comunidad y sus condiciones materiales de existencia (3-5 párrafos)",
  "valores_legitimados": ["valor1", "valor2", ...],
  "valores_deslegitimados": ["valor1", "valor2", ...],
  "emociones_facilitadas": ["emoción1", "emoción2", ...],
  "emociones_inhibidas": ["emoción1", "emoción2", ...],
  "orientacion_comportamiento": "qué acciones colectivas promueve esta narrativa (1 párrafo)"
}}
"""


def call_llm(user_prompt):
    for attempt in range(MAX_RETRIES):
        try:
            if LLM_PROVIDER == "claude":
                resp = client.messages.create(
                    model=MODEL, max_tokens=4096, temperature=TEMPERATURE,
                    system=SYSTEM_PROMPT_N5,
                    messages=[{"role": "user", "content": user_prompt}])
                text = resp.content[0].text.strip()
                tok_in, tok_out = resp.usage.input_tokens, resp.usage.output_tokens
            else:
                resp = client.chat.completions.create(
                    model=MODEL, temperature=TEMPERATURE, max_tokens=4096,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": SYSTEM_PROMPT_N5},
                              {"role": "user", "content": user_prompt}])
                text = resp.choices[0].message.content.strip()
                tok_in, tok_out = resp.usage.prompt_tokens, resp.usage.completion_tokens

            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            return json.loads(text), tok_in, tok_out
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(3 * (attempt + 1))
            else:
                print(f"  ⚠ Error LLM: {e}")
    return None, 0, 0


def build_regime_context(regime_row):
    """Construye el texto de contexto completo para un régimen."""
    rid = regime_row["ID_regimen"]

    # Escenarios
    esc_ids = df_esc_reg[df_esc_reg["ID_regimen"] == rid]["ID_escenario"].tolist()
    esc_lines = []
    for eid in esc_ids:
        esc_row = df_esc[df_esc["ID_escenario"] == eid]
        if len(esc_row) > 0:
            r = esc_row.iloc[0]
            sesgo_row = df_sesgo[df_sesgo["ID_escenario"] == eid]
            sesgo_text = ""
            if len(sesgo_row) > 0:
                sesgo_text = f" [+{sesgo_row.iloc[0].get('polo_positivo','')} / -{sesgo_row.iloc[0].get('polo_negativo','')}]"
            esc_lines.append(f"  • {r.get('nombre_escenario', eid)} (estatus: {r.get('estatus', '?')}){sesgo_text}")
    escenarios_text = "\n".join(esc_lines) if esc_lines else "  (no disponibles)"

    # Ejes valorativos
    ejes_rows = df_ejes[df_ejes["ID_regimen"] == rid]
    ejes_lines = [f"  • {r.get('polo_a', '')} ↔ {r.get('polo_b', '')} (var={r.get('varianza_explicada', 0):.1%})"
                   for _, r in ejes_rows.drop_duplicates(subset=["polo_a", "polo_b"]).iterrows()]
    ejes_text = "\n".join(ejes_lines) if ejes_lines else "  (no disponibles)"

    # Metáforas derivadas
    deriv_rows = df_deriv[df_deriv["ID_regimen"] == rid]
    deriv_lines = [f"  • {r.get('metafora_derivada', '')} [{r.get('tipo_inferencia', '')}]"
                    for _, r in deriv_rows.iterrows()]
    derivadas_text = "\n".join(deriv_lines) if deriv_lines else "  (ninguna)"

    # Posicionamiento
    pos_rows = df_pos[df_pos["ID_escenario"].isin(esc_ids)]
    pos_lines = [f"  • {r.get('grupo_social', '')}: {r.get('accion_atribuida', '')} ({r.get('legitimacion', '')})"
                  for _, r in pos_rows.head(15).iterrows()]
    posicionamiento_text = "\n".join(pos_lines) if pos_lines else "  (no disponible)"

    # Afectos
    af_rows = df_afecto[df_afecto["ID_escenario"].isin(esc_ids)]
    af_lines = [f"  • {r.get('emocion', '')} ({r.get('tipo', '')}): {r.get('funcion_social', '')}"
                 for _, r in af_rows.head(15).iterrows()]
    afectos_text = "\n".join(af_lines) if af_lines else "  (no disponible)"

    return escenarios_text, ejes_text, derivadas_text, posicionamiento_text, afectos_text


print("✓ Prompt y funciones configuradas")

In [ ]:
# ============================================================
# 5B. GENERAR NARRATIVA DOMINANTE
# ============================================================

total_tok_in = 0
total_tok_out = 0
t0 = time.time()

all_narratives = []

# Narrativa dominante
print(f"Generando narrativa dominante: {dominant_regime['nombre_regimen'][:50]}...")
esc_text, ejes_text, deriv_text, pos_text, af_text = build_regime_context(dominant_regime)

prompt = PROMPT_NARRATIVA.format(
    nombre_regimen=dominant_regime["nombre_regimen"],
    tipo="DOMINANTE",
    score=dominant_regime["score"],
    frecuencia=dominant_regime["frecuencia_agregada"],
    n_docs=dominant_regime.get("n_documentos", "?"),
    coherencia=dominant_regime["coherencia_interna"],
    dispersion=dominant_regime["dispersion_textual"],
    escenarios_text=esc_text,
    ejes_text=ejes_text,
    derivadas_text=deriv_text,
    posicionamiento_text=pos_text,
    afectos_text=af_text,
)

result, tok_in, tok_out = call_llm(prompt)
total_tok_in += tok_in
total_tok_out += tok_out

if result:
    result["ID_regimen"] = dominant_regime["ID_regimen"]
    result["tipo"] = "dominante"
    result["score"] = dominant_regime["score"]
    result["frecuencia_agregada"] = dominant_regime["frecuencia_agregada"]
    result["coherencia_interna"] = dominant_regime["coherencia_interna"]
    result["dispersion_textual"] = dominant_regime["dispersion_textual"]
    all_narratives.append(result)
    print(f"  ✓ Narrativa dominante: {result.get('nombre_narrativa', '?')}")
else:
    print("  ⚠ Error en la generación de la narrativa dominante")

time.sleep(RATE_LIMIT_PAUSE)

## 6. Narrativas alternativas

In [ ]:
# ============================================================
# 6. NARRATIVAS ALTERNATIVAS (challenger + emergente)
# ============================================================

alternativas = df_ranking[df_ranking["tipo"].isin(["challenger", "emergente"])]

for _, alt_row in alternativas.iterrows():
    print(f"Generando narrativa {alt_row['tipo']}: {alt_row['nombre_regimen'][:50]}...")
    esc_text, ejes_text, deriv_text, pos_text, af_text = build_regime_context(alt_row)

    prompt = PROMPT_NARRATIVA.format(
        nombre_regimen=alt_row["nombre_regimen"],
        tipo=alt_row["tipo"].upper(),
        score=alt_row["score"],
        frecuencia=alt_row["frecuencia_agregada"],
        n_docs=alt_row.get("n_documentos", "?"),
        coherencia=alt_row["coherencia_interna"],
        dispersion=alt_row["dispersion_textual"],
        escenarios_text=esc_text,
        ejes_text=ejes_text,
        derivadas_text=deriv_text,
        posicionamiento_text=pos_text,
        afectos_text=af_text,
    )

    result, tok_in, tok_out = call_llm(prompt)
    total_tok_in += tok_in
    total_tok_out += tok_out

    if result:
        result["ID_regimen"] = alt_row["ID_regimen"]
        result["tipo"] = alt_row["tipo"]
        result["score"] = alt_row["score"]
        result["frecuencia_agregada"] = alt_row["frecuencia_agregada"]
        result["coherencia_interna"] = alt_row["coherencia_interna"]
        result["dispersion_textual"] = alt_row["dispersion_textual"]
        all_narratives.append(result)
        print(f"  ✓ {alt_row['tipo']}: {result.get('nombre_narrativa', '?')}")
    else:
        print("  ⚠ Error")

    time.sleep(RATE_LIMIT_PAUSE)

elapsed = time.time() - t0
costo = (total_tok_in / 1e6 * (3 if LLM_PROVIDER == "claude" else 2.5)) + \
        (total_tok_out / 1e6 * (15 if LLM_PROVIDER == "claude" else 10))

print(f"\n✓ Narrativas generadas: {len(all_narratives)}")
print(f"  Tokens: {total_tok_in:,} in + {total_tok_out:,} out")
print(f"  Costo: ${costo:.2f} USD")
print(f"  Tiempo: {elapsed:.1f}s")

## 7. Muestra para evaluación human-in-the-loop

Todas las narrativas generadas se exportan para revisión del analista, quien evalúa:
- ¿La narrativa es coherente con los escenarios del régimen?
- ¿Los valores legitimados/deslegitimados corresponden a los sesgos evaluativos?
- ¿La orientación del comportamiento se sustenta en las evidencias del corpus?

In [ ]:
# ============================================================
# 7. MUESTRA HUMAN-IN-THE-LOOP
# ============================================================

eval_rows = []
for narr in all_narratives:
    eval_rows.append({
        "tipo": narr.get("tipo", ""),
        "ID_regimen": narr.get("ID_regimen", ""),
        "nombre_narrativa": narr.get("nombre_narrativa", ""),
        "descripcion_resumen": str(narr.get("descripcion_narrativa", ""))[:500] + "...",
        "valores_legitimados": json.dumps(narr.get("valores_legitimados", []), ensure_ascii=False),
        "emociones_facilitadas": json.dumps(narr.get("emociones_facilitadas", []), ensure_ascii=False),
        "orientacion_comportamiento": str(narr.get("orientacion_comportamiento", ""))[:300],
        "score": narr.get("score", 0),
        # Columnas para evaluación manual
        "narrativa_coherente": "",
        "valores_correctos": "",
        "orientacion_sustentada": "",
        "observaciones": "",
    })

df_eval = pd.DataFrame(eval_rows)
eval_path = os.path.join(OUTPUT_DIR, "n5_muestra_evaluacion.csv")
df_eval.to_csv(eval_path, index=False, encoding='utf-8-sig')
print(f"✓ {eval_path} ({len(df_eval)} narrativas para evaluar)")
print("  Columnas: narrativa_coherente, valores_correctos, orientacion_sustentada, observaciones")

## 8. Exportación

In [ ]:
# ============================================================
# 8A. CSVs DE SALIDA
# ============================================================

# n5_narrativa_cultural.csv — solo la dominante
dominant_narr = [n for n in all_narratives if n.get("tipo") == "dominante"]
if dominant_narr:
    dn = dominant_narr[0]
    df_dominant = pd.DataFrame([{
        "ID_narrativa": "NAR-001",
        "ID_regimen_dominante": dn["ID_regimen"],
        "nombre_narrativa": dn.get("nombre_narrativa", ""),
        "descripcion_narrativa": dn.get("descripcion_narrativa", ""),
        "score_dominancia": dn["score"],
        "frecuencia_agregada": dn["frecuencia_agregada"],
        "coherencia_interna": dn["coherencia_interna"],
        "dispersion_textual": dn["dispersion_textual"],
        "valores_legitimados": json.dumps(dn.get("valores_legitimados", []), ensure_ascii=False),
        "valores_deslegitimados": json.dumps(dn.get("valores_deslegitimados", []), ensure_ascii=False),
        "emociones_facilitadas": json.dumps(dn.get("emociones_facilitadas", []), ensure_ascii=False),
        "emociones_inhibidas": json.dumps(dn.get("emociones_inhibidas", []), ensure_ascii=False),
        "orientacion_comportamiento": dn.get("orientacion_comportamiento", ""),
    }])
    df_dominant.to_csv(os.path.join(OUTPUT_DIR, "n5_narrativa_cultural.csv"),
                        index=False, encoding='utf-8-sig')
    print("✓ n5_narrativa_cultural.csv")

# n5_narrativas_alternativas.csv — todas
alt_rows = []
for i, narr in enumerate(all_narratives):
    alt_rows.append({
        "ID_narrativa": f"NAR-{i+1:03d}",
        "ID_regimen": narr["ID_regimen"],
        "nombre_narrativa": narr.get("nombre_narrativa", ""),
        "descripcion": narr.get("descripcion_narrativa", ""),
        "score": narr["score"],
        "tipo": narr["tipo"],
        "valores_legitimados": json.dumps(narr.get("valores_legitimados", []), ensure_ascii=False),
        "valores_deslegitimados": json.dumps(narr.get("valores_deslegitimados", []), ensure_ascii=False),
        "emociones_facilitadas": json.dumps(narr.get("emociones_facilitadas", []), ensure_ascii=False),
        "emociones_inhibidas": json.dumps(narr.get("emociones_inhibidas", []), ensure_ascii=False),
        "orientacion_comportamiento": narr.get("orientacion_comportamiento", ""),
    })
df_all_narr = pd.DataFrame(alt_rows)
df_all_narr.to_csv(os.path.join(OUTPUT_DIR, "n5_narrativas_alternativas.csv"),
                     index=False, encoding='utf-8-sig')
print(f"✓ n5_narrativas_alternativas.csv ({len(df_all_narr)} narrativas)")

In [ ]:
# ============================================================
# 8B. RESUMEN DEL CAMPO METAFÓRICO (JSON para dashboard)
# ============================================================

resumen = {
    "corpus": "Informe Final de la Comisión de la Verdad de Colombia",
    "timestamp": datetime.datetime.now().isoformat(),
    "estadisticas_generales": {
        "metaforas_primarias": len(df_n1),
        "metaforas_convencionales": len(df_conv),
        "escenarios": len(df_esc),
        "regimenes": len(df_reg),
        "narrativas_generadas": len(all_narratives),
    },
    "ranking_regimenes": df_ranking[["ID_regimen", "nombre_regimen", "tipo", "score",
                                       "frecuencia_agregada", "coherencia_interna",
                                       "dispersion_textual"]].to_dict(orient="records"),
    "narrativa_dominante": dominant_narr[0] if dominant_narr else None,
    "narrativas_alternativas": [n for n in all_narratives if n.get("tipo") != "dominante"],
}

with open(os.path.join(OUTPUT_DIR, "n5_resumen_campo_metaforico.json"), 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print("✓ n5_resumen_campo_metaforico.json")

In [ ]:
# ============================================================
# 8C. METADATOS Y RESUMEN FINAL
# ============================================================

metadata = {
    "nivel": "N5",
    "timestamp": datetime.datetime.now().isoformat(),
    "versions": {pkg: getattr(importlib.import_module(pkg), "__version__", "?")
                  for pkg in ["pandas", "numpy", "anthropic", "openai"]
                  if importlib.util.find_spec(pkg)},
    "llm_provider": LLM_PROVIDER,
    "model": MODEL,
    "temperature": TEMPERATURE,
    "pesos_ranking": {"frecuencia": W_FRECUENCIA, "coherencia": W_COHERENCIA, "dispersion": W_DISPERSION},
    "narrativas_generadas": len(all_narratives),
    "tokens_in": total_tok_in,
    "tokens_out": total_tok_out,
    "costo_usd": round(costo, 2),
    "tiempo_seg": round(elapsed, 1),
}
with open(os.path.join(OUTPUT_DIR, "n5_metadata.json"), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print("N5 — NARRATIVA CULTURAL COMPLETADO")
print(f"{'='*60}")
print(f"\n  Narrativa dominante: {dominant_narr[0].get('nombre_narrativa', '?') if dominant_narr else 'N/A'}")
print(f"  Narrativas alternativas: {len(all_narratives) - 1}")
print(f"  Costo total: ${costo:.2f} USD")
print(f"\n  Archivos en {OUTPUT_DIR}:")
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f_name)) / 1024
    print(f"    📄 {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: N5_narrativa_cultural_viz.ipynb para dashboard final")